In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
import scipy.io as sio
from collections import defaultdict 
from kilosort.io import load_ops
from kilosort.data_tools import (
    mean_waveform, cluster_templates, get_cluster_spikes,
    get_spike_waveforms, get_best_channel
)

# Analyze the basic waveform properties
def analyze_waveform(waveform, fs=30000):
    min_idx = np.argmin(waveform)
    max_idx = np.argmax(waveform[min_idx:]) + min_idx
    trough_to_peak_time = (max_idx - min_idx) / fs * 1000  # ms
    return {
        "min_voltage": waveform[min_idx],
        "max_voltage": waveform[max_idx],
        "trough_to_peak_time_ms": trough_to_peak_time
    }

# Create ISI histogram and record peak in 1s window
def plot_isi(spike_times, fs, save_path):
    times_sec = spike_times / fs
    isi = np.diff(times_sec)
    plt.figure()
    plt.hist(isi, bins=np.arange(0, 1.05, 0.05))
    plt.title("ISI Histogram (50 ms bins)")
    plt.xlabel("ISI (s)")
    plt.ylabel("Count")
    plt.savefig(os.path.join(save_path, "isi_histogram.png"))
    plt.close()
    
    isi_fine = np.histogram(isi, bins=np.arange(0, 1.0, 0.01))[0]
    max_peak = np.max(isi_fine)
    with open(os.path.join(save_path, "isi_max_peak.txt"), "w") as f:
        f.write(f"Max ISI count (1s window, 10ms bin): {max_peak}\n")

# Main per-cluster function: handles waveform and ISI processing
def process_cluster(cluster_id, session_name, results_dir, fs=30000):
    results_dir = Path(results_dir)
    save_dir = results_dir / f"{session_name}_cluster{cluster_id}"
    save_dir.mkdir(parents=True, exist_ok=True)
    print(results_dir)
    
    ops = load_ops(results_dir / 'ops.npy')
    
    t = (np.arange(ops['nt']) / ops['fs']) * 1000  # ms

    # Plot using mean_waveform and cluster_templates
    mean_wv, spike_subset = mean_waveform(cluster_id, results_dir, n_spikes=100, best=True)
    mean_temp = cluster_templates(cluster_id, results_dir, mean=True,
                                  best=True, spike_subset=spike_subset)
    best_chan = get_best_channel(results_dir, cluster_id) +1

    plt.figure()
    plt.plot(t, mean_wv, c='black', linestyle='dashed', linewidth=2, label='mean waveform')
    plt.plot(t, mean_temp, linewidth=1, label='template')
    plt.title(f"Cluster {cluster_id} – Best channel: {best_chan}")
    plt.xlabel("Time (ms)")
    plt.legend()
    plt.savefig(save_dir / "waveform_template_mean.png")
    plt.close()

    # Plot from raw waveform average
    

    # Analyze both waveform types
    metrics = {
        "from_mean_waveform": analyze_waveform(mean_wv, fs),
        
    }
    with open(save_dir / "waveform_info.txt", "w") as f:
        for source, vals in metrics.items():
            f.write(f"{source}:\n")
            for k, v in vals.items():
                f.write(f"  {k}: {v:.3f}\n")
            f.write("\n")

    # Compute ISI and save
    spikes = np.load(results_dir / 'spike_times.npy')
    cluster_labels = np.load(results_dir / 'spike_clusters.npy')
    cluster_spikes = spikes[cluster_labels == cluster_id]
    plot_isi(cluster_spikes, fs, save_dir)

    # Save timestamps in MATLAB .mat format
    ts_dir = results_dir / "timestamps"
    ts_dir.mkdir(exist_ok=True)
    mat_filename = ts_dir / f"{cluster_id}.mat"
    sio.savemat(mat_filename, {f"cluster_{cluster_id}_timestamp": cluster_spikes[:, np.newaxis]})
    return best_chan
# Top-level function: reads spreadsheet and loops through all columns
def kilosort_postprocess(excel_path):
    df = pd.read_excel(excel_path, header=None)
    best_rows_by_dir = defaultdict(list)
    for col in df.columns:
        session_name = str(df.iloc[0, col])
        results_path = str(df.iloc[1, col])
        cluster_ids = df.iloc[2:, col].dropna()

        for cluster_id in cluster_ids:
            cluster_id = int(cluster_id)
            print(f"Processing {session_name}, Cluster {cluster_id}")
            try:
                best_chan = process_cluster(cluster_id, session_name, results_path)  
                best_rows_by_dir[Path(results_path)].append({                         
                    "session": session_name,
                    "cluster_id": cluster_id,
                    "best_channel": best_chan
                })
            except Exception as e:
                print(f"Error processing {session_name} Cluster {cluster_id}: {e}")

    for rdir, rows in best_rows_by_dir.items():  # NEW
        if not rows:
            continue
        out_df = pd.DataFrame(rows).sort_values(["session", "cluster_id"]).reset_index(drop=True)
        out_df.to_excel(Path(rdir) / "best_channels.xlsx", index=False)


In [17]:
feedersheet_dir= Path(r"D:\Data\PTEN Project\feedersheets\remaining_waveform.xlsx")
kilosort_postprocess(feedersheet_dir)

Processing m5s2, Cluster 15
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 16
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 18
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 6
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 29
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 60
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 62
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert_False
Processing m5s2, Cluster 66
D:\Data\PTEN Project\PTEN\M5_s2\2023-11-16_14-11-05\binary\kilosort_tl6_tu7_fs30000.0invert